# PlantVillage - Fine-Tuning do EfficientNetB0

Este notebook reúne o refinamento feito a partir do modelo base salvo. A intenção é documentar o experimento de fine-tuning parcial, mantendo BatchNormalization congelada e usando learning rate baixo para adaptar o topo do backbone sem destruir o conhecimento pré-treinado.


## 1. Imports e configuração

Mesmos imports principais do baseline para reconstruir o pipeline e avaliar o modelo refinado.


In [1]:
import os 
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import random 
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from collections import Counter 
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
import seaborn as sns

## 2. Reconstrução dos dados

O fine-tuning precisa usar exatamente o mesmo split e o mesmo mapeamento de classes do baseline. Por isso, mantemos `random_state=42`, split estratificado e `class_names = sorted(set(labels))`.


In [2]:
def load_data(path):
    files = []
    labels = []

    for label in os.listdir(path):
        label_path = os.path.join(path, label)
        if os.path.isdir(label_path):
            for file in os.listdir(label_path):
                files.append(os.path.join(label_path, file))
                labels.append(label)    
    
    return files, labels 

In [3]:
path = os.path.join('color')
files, labels = load_data(path)

train_files, temp_files, train_labels, temp_labels = train_test_split(
    files,
    labels,
    test_size=0.2,
    random_state=42, 
    stratify=labels
)

val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files, 
    temp_labels,
    test_size=0.5,
    random_state=42, 
    stratify=temp_labels
)

In [4]:
class_names = sorted(set(labels))

class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}
num_classes = len(class_names)

print("Número de classes:", len(class_names))
print(class_to_idx)

Número de classes: 38
{'Apple___Apple_scab': 0, 'Apple___Black_rot': 1, 'Apple___Cedar_apple_rust': 2, 'Apple___healthy': 3, 'Blueberry___healthy': 4, 'Cherry_(including_sour)___Powdery_mildew': 5, 'Cherry_(including_sour)___healthy': 6, 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 7, 'Corn_(maize)___Common_rust_': 8, 'Corn_(maize)___Northern_Leaf_Blight': 9, 'Corn_(maize)___healthy': 10, 'Grape___Black_rot': 11, 'Grape___Esca_(Black_Measles)': 12, 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 13, 'Grape___healthy': 14, 'Orange___Haunglongbing_(Citrus_greening)': 15, 'Peach___Bacterial_spot': 16, 'Peach___healthy': 17, 'Pepper,_bell___Bacterial_spot': 18, 'Pepper,_bell___healthy': 19, 'Potato___Early_blight': 20, 'Potato___Late_blight': 21, 'Potato___healthy': 22, 'Raspberry___healthy': 23, 'Soybean___healthy': 24, 'Squash___Powdery_mildew': 25, 'Strawberry___Leaf_scorch': 26, 'Strawberry___healthy': 27, 'Tomato___Bacterial_spot': 28, 'Tomato___Early_blight': 29, 'Tomato___Lat

In [5]:
train_label_ids = [class_to_idx[label] for label in train_labels]
val_label_ids = [class_to_idx[label] for label in val_labels]
test_label_ids = [class_to_idx[label] for label in test_labels]

## 3. Reconstrução do pipeline `tf.data`

O pipeline deve ser igual ao usado no baseline para que a comparação seja justa.


In [6]:
img_size = (224, 224)
batch_size = 16

def load_image(image_path, label_id): 
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, img_size)
    image = tf.cast(image, tf.float32)
   
    return image, label_id

In [7]:
train_ds = tf.data.Dataset.from_tensor_slices((train_files, train_label_ids))
val_ds = tf.data.Dataset.from_tensor_slices((val_files, val_label_ids))
test_ds = tf.data.Dataset.from_tensor_slices((test_files, test_label_ids))

In [8]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    train_ds
    .shuffle(buffer_size=4096, seed=42)
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(batch_size)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_ds
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(batch_size)
    .prefetch(AUTOTUNE)
)

test_ds = (
    test_ds
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(batch_size)
    .prefetch(AUTOTUNE)
)

In [9]:
for images, targets in train_ds.take(1):
    print(images.shape)
    print(targets.shape)
    print(targets[:10])

(16, 224, 224, 3)
(16,)
tf.Tensor([33 27 19 24 24 19  1  3 13 28], shape=(10,), dtype=int32)


## 4. Carregamento do baseline e descongelamento parcial

O modelo base é carregado de `models/efficientnet_b0_baseline.keras`. Em seguida, apenas as últimas camadas do backbone são liberadas para treino. Isso torna o refinamento mais leve e reduz risco de overfitting.


In [10]:
model = tf.keras.models.load_model("models/efficientnet_b0_baseline.keras")  
base_model = model.layers[1]     
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

## 5. Congelamento de BatchNormalization

BatchNormalization costuma ser instável em fine-tuning com batch pequeno. Mantê-la congelada preserva as estatísticas aprendidas no pré-treinamento.


In [11]:
for layer in base_model.layers: 
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

## 6. Recompilação com learning rate baixo

Sempre recompile após alterar `trainable`. No fine-tuning, o learning rate deve ser baixo para adaptar o modelo gradualmente.


In [12]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 7. Callbacks

`EarlyStopping` evita continuar treinando quando a validação para de melhorar. `ReduceLROnPlateau` reduz o learning rate se o progresso estagnar.


In [13]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7
    )
]

## 8. Treino refinado

Este experimento refinou o modelo por até 5 épocas. Como seu computador falhou acima disso, a abordagem mais segura daqui para frente é treinar em blocos curtos, salvar checkpoint e reiniciar o kernel quando necessário.


In [ ]:
fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)

model.save("models/efficientnet_b0_fine_tuned_5_epochs.keras")

## 9. Avaliação do fine-tuning

A comparação correta é contra o baseline, olhando não sé acurácia, mas `macro F1`, recall das classes pequenas e possíveis regressões por classe.


In [15]:
model = tf.keras.models.load_model("models/efficientnet_b0_fine_tuned_5_epochs.keras")
y_true = []
y_pred = []

for images, labels_batch in val_ds:
    preds = model.predict(images, verbose=0)
    pred_classes = np.argmax(preds, axis=1)

    y_true.extend(labels_batch.numpy())
    y_pred.extend(pred_classes)

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
))

                                                    precision    recall  f1-score   support

                                Apple___Apple_scab     0.9839    0.9683    0.9760        63
                                 Apple___Black_rot     0.9841    1.0000    0.9920        62
                          Apple___Cedar_apple_rust     1.0000    1.0000    1.0000        28
                                   Apple___healthy     0.9878    0.9878    0.9878       164
                               Blueberry___healthy     1.0000    1.0000    1.0000       150
          Cherry_(including_sour)___Powdery_mildew     1.0000    0.9905    0.9952       105
                 Cherry_(including_sour)___healthy     1.0000    1.0000    1.0000        86
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot     0.9318    0.7885    0.8542        52
                       Corn_(maize)___Common_rust_     1.0000    1.0000    1.0000       119
               Corn_(maize)___Northern_Leaf_Blight     0.8972    0.9697    0.93

In [16]:
val_loss, val_accuracy = model.evaluate(val_ds)
print(val_loss, val_accuracy)

340/340 ━━━━━━━━━━━━━━━━━━━━ 140s 403ms/step - accuracy: 0.9877 - loss: 0.0334
0.033397119492292404 0.9876611232757568
